Imports and Initialization

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix, 
                             roc_auc_score, roc_curve)

# Styling and directory setups
sns.set_theme(style="whitegrid")
os.makedirs('charts', exist_ok=True)
print("Libraries loaded. Workspace ready for HR Analytics pipeline!")

Libraries loaded. Workspace ready for HR Analytics pipeline!


Task 1 — Data Loading & Exploration

In [3]:
# Load dataset (adjust filename to match your local filename if needed)
df = pd.read_csv('WA_Fn-UseC_-HR-Employee-Attrition.csv')

print("--- Dataset Dimensions ---")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}\n")

print("--- First 3 Rows Preview ---")
display(df.head(3))

# Calculate Attrition Rate
attrition_counts = df['Attrition'].value_counts()
attrition_rate = (attrition_counts['Yes'] / df.shape[0]) * 100

print("\n--- Attrition Target Distribution ---")
print(f"Employees Stayed (No): {attrition_counts['No']}")
print(f"Employees Left (Yes): {attrition_counts['Yes']}")
print(f"Base Attrition Rate: {attrition_rate:.2f}%")

# Count structural datatypes
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"\nNumeric features count: {len(numeric_cols)}")
print(f"Categorical features count: {len(categorical_cols)}")

print("\n--- Task 1 Observation ---")
print("Observation: The dataset exhibits a significant class imbalance. Roughly 16.12% of the employees ")
print("have left, meaning the baseline model could achieve 84% accuracy by simply guessing 'No' for everyone.")
print("We must adjust model class weights to account for this skew.")

--- Dataset Dimensions ---
Rows: 1470, Columns: 35

--- First 3 Rows Preview ---


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0



--- Attrition Target Distribution ---
Employees Stayed (No): 1233
Employees Left (Yes): 237
Base Attrition Rate: 16.12%

Numeric features count: 26
Categorical features count: 9

--- Task 1 Observation ---
Observation: The dataset exhibits a significant class imbalance. Roughly 16.12% of the employees 
have left, meaning the baseline model could achieve 84% accuracy by simply guessing 'No' for everyone.
We must adjust model class weights to account for this skew.


C:\Users\Asus\AppData\Local\Temp\ipykernel_16432\584099526.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object']).columns.tolist()


Task 2 — Data Cleaning & Preprocessing

In [4]:
# 1. Verify missing values
print("Total Missing Values:", df.isnull().sum().sum())

# 2. Drop low-variance/irrelevant columns
cols_to_drop = ['EmployeeNumber', 'Over18', 'StandardHours', 'EmployeeCount']
df_cleaned = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

# 3. Target mapping
df_cleaned['Attrition'] = df_cleaned['Attrition'].map({'Yes': 1, 'No': 0})

# 4. Separate target and features
X = df_cleaned.drop(columns=['Attrition'])
y = df_cleaned['Attrition']

# Identify remaining categorical columns for encoding (Updated to support Pandas 3/4 string transitions)
cat_features = X.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
print(f"Categorical features detected for encoding: {cat_features}")

# 5. One-Hot Encoding (Enforce integer 1/0 formatting explicitly)
X_encoded = pd.get_dummies(X, columns=cat_features, drop_first=True, dtype=int)

# 6. Feature Scaling
scaler = StandardScaler()
num_features = X.select_dtypes(include=[np.number]).columns.tolist()
X_scaled = X_encoded.copy()
X_scaled[num_features] = scaler.fit_transform(X_encoded[num_features])

print(f"\nFinal feature dimensions after cleaning & encoding: {X_scaled.shape}")

Total Missing Values: 0
Categorical features detected for encoding: ['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']

Final feature dimensions after cleaning & encoding: (1470, 44)


Task 3 — Exploratory Data Analysis & Feature Mining

In [5]:
# 1. Attrition Rate by Department
print("1. ATTRITION RATE BY DEPARTMENT:")
dept_attrition = df.groupby('Department')['Attrition'].value_counts(normalize=True).unstack() * 100
dept_attrition['Total_Employees'] = df['Department'].value_counts()
display(dept_attrition.round(2))

# 2. Attrition Rate by Job Role
print("\n2. TOP 5 JOB ROLES WITH HIGHEST EXIT RATES:")
role_attrition = df.groupby('JobRole')['Attrition'].value_counts(normalize=True).unstack() * 100
role_attrition['Total_Employees'] = df['JobRole'].value_counts()
display(role_attrition.sort_values(by='Yes', ascending=False).head(5).round(2))

# 3. Attrition vs Monthly Income
print("\n3. INCOME DISPARITY PROFILE (AVERAGE MONTHLY INCOME):")
income_profile = df.groupby('Attrition')['MonthlyIncome'].agg(['mean', 'median', 'std'])
display(income_profile.round(2))

# 4. Attrition vs Work-Life Balance Rating
print("\n4. ATTRITION RATE BY WORK-LIFE BALANCE RATING (1=Bad, 4=Best):")
wlb_attrition = df.groupby('WorkLifeBalance')['Attrition'].value_counts(normalize=True).unstack() * 100
display(wlb_attrition.round(2))

# 5. Attrition vs Years at Company (Tenure)
print("\n5. AVERAGE TENURE PROFILE (YEARS AT COMPANY):")
tenure_profile = df.groupby('Attrition')['YearsAtCompany'].agg(['mean', 'median'])
display(tenure_profile.round(2))

1. ATTRITION RATE BY DEPARTMENT:


Attrition,No,Yes,Total_Employees
Department,,,
Human Resources,80.95,19.05,63
Research & Development,86.16,13.84,961
Sales,79.37,20.63,446



2. TOP 5 JOB ROLES WITH HIGHEST EXIT RATES:


Attrition,No,Yes,Total_Employees
JobRole,,,
Sales Representative,60.24,39.76,83
Laboratory Technician,76.06,23.94,259
Human Resources,76.92,23.08,52
Sales Executive,82.52,17.48,326
Research Scientist,83.90,16.10,292



3. INCOME DISPARITY PROFILE (AVERAGE MONTHLY INCOME):


,mean,median,std
Attrition,,,
No,6832.74,5204.0,4818.21
Yes,4787.09,3202.0,3640.21



4. ATTRITION RATE BY WORK-LIFE BALANCE RATING (1=Bad, 4=Best):


Attrition,No,Yes
WorkLifeBalance,,
1,68.75,31.25
2,83.14,16.86
3,85.78,14.22
4,82.35,17.65



5. AVERAGE TENURE PROFILE (YEARS AT COMPANY):


,mean,median
Attrition,,
No,7.37,6.0
Yes,5.13,3.0


Task 4 & 5 — Model Validation & Matrix Extraction

In [6]:
# Train-Test Split (80/20, stratified to preserve the 16.12% attrition imbalance)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set shape: {X_train.shape}, Test set shape: {X_test.shape}")

# Initialize models with balance handling parameters
models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(class_weight='balanced', random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

results_dict = {}

# Fit and capture performance
for name, model in models.items():
    if name == 'Gradient Boosting':
        # Apply manual scale weights for Gradient Boosting since it lacks class_weight native args
        scale_weight = (len(y_train) - sum(y_train)) / sum(y_train)
        sample_weights = np.where(y_train == 1, scale_weight, 1)
        model.fit(X_train, y_train, sample_weight=sample_weights)
    else:
        model.fit(X_train, y_train)
        
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    
    # Capture metrics
    report = classification_report(y_test, preds, output_dict=True)
    auc = roc_auc_score(y_test, probs)
    
    results_dict[name] = {
        'Precision (Left)': f"{report['1']['precision']:.4f}",
        'Recall (Left)': f"{report['1']['recall']:.4f}",
        'F1-Score (Left)': f"{report['1']['f1-score']:.4f}",
        'ROC-AUC Score': f"{auc:.4f}"
    }

# Display Model Comparison Matrix Table
performance_df = pd.DataFrame(results_dict).T
print("\n--- Model Performance Comparison Matrix ---")
display(performance_df)

Training set shape: (1176, 44), Test set shape: (294, 44)

--- Model Performance Comparison Matrix ---


,Precision (Left),Recall (Left),F1-Score (Left),ROC-AUC Score
Logistic Regression,0.3563,0.6596,0.4627,0.8036
Random Forest,0.4688,0.3191,0.3797,0.7712
Gradient Boosting,0.4074,0.4681,0.4356,0.7796


Task 5 — Model Evaluation & Feature Extraction

In [ ]:
# Isolate the winning model (Logistic Regression)
best_model = models['Logistic Regression']
lr_preds = best_model.predict(X_test)
lr_probs = best_model.predict_proba(X_test)[:, 1]

# 1. Output Confusion Matrix
cm = confusion_matrix(y_test, lr_preds)
print("--- Confusion Matrix Breakdown ---")
print(f"True Negatives (Predicted Stay, Actually Stayed): {cm[0][0]}")
print(f"False Positives (Predicted Leave, Actually Stayed): {cm[0][1]}")
print(f"False Negatives (Predicted Stay, Actually Left): {cm[1][0]}")
print(f"True Positives (Predicted Leave, Actually Left): {cm[1][1]}")
print(f"Total Array:\n{cm}\n")

# 2. Extract Feature Importance using Logistic Regression Coefficients
coefficients = best_model.coef_[0]
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': np.abs(coefficients),
    'Direction': np.where(coefficients > 0, 'Drives Exit (Risk Factor)', 'Drives Retention (Anchor)')
}).sort_values(by='Importance', ascending=False)

print("--- Top 10 Most Critical Drivers of Employee Attrition ---")
display(feature_importance.head(10))

# State the best model reason clearly
print("\n--- Model Selection Analysis ---")
print("Winner: Logistic Regression")
print("Reasoning: In employee retention, missing a flight risk (False Negative) is much more costly than")
print("flagging a loyal employee (False Positive). Logistic Regression captured the highest Recall (65.96%)")
print("and the best overall ROC-AUC frontier (0.8036), making it the most actionable system for an HR team.")